# Kangri Cancer Screening — Train on Google Colab (Free GPU)

Use this notebook if training on your own laptop GPU runs out of memory, or if you just don't have a GPU handy. Colab gives you a free (but time-limited) GPU in the cloud.

**Before you run anything, turn on the GPU:**
1. Click the **Runtime** menu at the top.
2. Click **Change runtime type**.
3. Under "Hardware accelerator", pick **T4 GPU**.
4. Click **Save**.

Then run each cell below top to bottom (click the little play button on the left of each cell, or press Shift+Enter).

## Step 1 — Connect your Google Drive

This lets the notebook save training checkpoints to your Drive. That matters because Colab sessions can time out or disconnect — if that happens mid-training, your progress is still safe on Drive and you won't have to start over.

Running the cell below will pop up a permission request — click **Connect to Google Drive**, choose your account, and allow access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/kangri_cancer_checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', DRIVE_CHECKPOINT_DIR)

## Step 2 — Clone the project repo

This downloads all the project code (data prep scripts, model definition, etc.) into the Colab machine.

In [ ]:
%cd /content
!git clone https://github.com/bluetreesandredskies/kangri-cancer-screening.git
%cd kangri-cancer-screening

## Step 3 — Install the required packages

Colab already comes with PyTorch pre-installed, so this just adds the extra libraries the project needs.

In [ ]:
!pip install -q timm pyyaml matplotlib seaborn scikit-learn tqdm pillow pandas

## Step 4 — Get your processed dataset onto the Colab machine

The `data/processed/` folder (train/val/test images) is normally too big to keep committed to git. The easiest path: zip up your local `data/processed` folder, upload the zip to your Google Drive, then unzip it here.

Change `DRIVE_ZIP_PATH` below to wherever you uploaded the zip in your Drive.

In [ ]:
DRIVE_ZIP_PATH = '/content/drive/MyDrive/kangri_cancer_data/processed.zip'

import os
if os.path.exists(DRIVE_ZIP_PATH):
    !mkdir -p data
    !unzip -q "$DRIVE_ZIP_PATH" -d data/
    print('Unzipped dataset into data/processed')
else:
    print(f'Could not find {DRIVE_ZIP_PATH} — upload your zipped data/processed folder to Drive first, then update the path above and re-run this cell.')

## Step 5 — Point the config and checkpoints at Google Drive

This copies the repo's config so we can tweak it without editing the git-tracked file, and makes sure checkpoints get written straight to Drive as training runs.

In [ ]:
import shutil
os.makedirs('model/checkpoints', exist_ok=True)

# symlink the checkpoints folder to Drive so every save lands there directly
if os.path.islink('model/checkpoints') or os.path.exists('model/checkpoints'):
    shutil.rmtree('model/checkpoints', ignore_errors=True)
os.symlink(DRIVE_CHECKPOINT_DIR, 'model/checkpoints')
print('model/checkpoints now points at your Drive folder.')

## Step 6 — Confirm the GPU is actually being used

If this prints `False`, go back to the top and redo the Runtime > Change runtime type > T4 GPU step, then re-run all cells from the top.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))

## Step 7 — Train

This runs the exact same training script (`model/train.py`) you'd run locally — same config, same logic, just on Colab's GPU. Checkpoints save straight to your Drive folder as training goes, so a disconnect won't lose your progress.

In [ ]:
!python model/train.py --device cuda --data-dir data/processed

## Step 8 — Verify the checkpoint saved to Drive

You should see `best_model.pt` listed below. If your session ever disconnects, you can come back, redo steps 1–6, and it will still be there.

In [ ]:
!ls -la "$DRIVE_CHECKPOINT_DIR"